[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [Pandas, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)

# Missing Data


## What you will be able to do

Find the missing values in a table, tell `NaN`, `None` and `pd.NA` apart, and choose between
dropping, filling and leaving them, knowing what each choice does to the numbers you report.

## The idea

### The problem

Svalbard's data logger sent nothing on 2 March, so the readings have 24 hours with no temperature.
Real data has gaps like that everywhere: a sensor fails, a survey question is skipped, a join finds
no match, a spreadsheet cell is left empty. Every calculation that touches those rows has to do
something with them, and each choice gives a different answer.

Count the missing hours as zero degrees and Svalbard's March comes out warmer than it was. Add them
up as a NumPy array and the total is missing too, while pandas' own `sum` skips them and gives the
total you would get by counting them as zero. Copy the last reading forward and the table holds 24
readings that nobody took, which look exactly like the real ones. None of these raises an error, so
the choice has to be made on purpose, and the first step is knowing which values are missing and
what pandas already does with them.

pandas makes several of those choices for you by default. A mean leaves missing values out, a
comparison with one is false for every operator but `!=`, and `groupby` drops a group whose key is
missing. Those defaults are usually what you want, and the cases where they are not are the ones
this notebook is about.

### What a missing value is

> A **missing value** is a position in a Series or DataFrame marked as having no value. pandas marks
> it with `NaN`, the floating point "not a number", in a column of decimals, with `NaT` in a column
> of dates and times, with `None` or `NaN` in a column of Python objects such as text, and with
> `pd.NA` in the nullable types such as `Int64`. `isna` and `notna` find every one of them.

### Why it works that way

`NaN` is a float, a special value that floating point arithmetic defines, so a column of decimals
can hold it without changing type. A column of whole numbers cannot, and pandas turns such a column
into decimals the moment one value goes missing, which is why 1, 2 and 3 can come back as 1.0, 2.0
and 3.0. The nullable `Int64` type keeps whole numbers whole by marking the gap with `pd.NA`
instead.

`NaN` is not equal to anything, itself included, so `== np.nan` can never find it, and every
comparison with it is false apart from `!=`, which is true. `pd.NA` goes further. A comparison with
it is `<NA>` again, missing in turn, and asking whether that is true raises an error, because pandas
will not guess.

Reductions such as `mean` and `sum` skip missing values unless given `skipna=False`, and `count`
counts only the values that are there. `dropna` removes the rows or columns that have them, and
`fillna`, `ffill`, `bfill` and `interpolate` put a value in each gap. Every fill is an estimate of a
reading that was never taken, so it belongs to the analysis, and the table should say somewhere that
it was made.

### Where this shows up

The **Boolean Filtering** notebook found two masks that disagreed by Svalbard's 24 hours, because of
how comparisons treat `NaN`. The **Reading Data** notebook turns markers such as an empty cell into
`NaN` as a file loads, the **Merging** notebook gets `NaN` for a station with no readings, and the
**Reshaping** notebook carries Svalbard's 24 gaps into a wide table and shows that `stack` drops
them by default. The **groupby** notebook counts with `size` and `count`, which differ by exactly
the missing values.

### What this notebook covers

- finding the missing readings with `isna`, and counting them by station
- `NaN`, `None` and `pd.NA`, and what a missing value does to a column's type
- what `mean`, `sum` and `count` do with the gaps
- `dropna` with `how`, `subset` and `thresh`
- filling with a value, with `ffill` and `bfill`, and with `interpolate`, and what each does to a
  mean
- four errors, one of them a comparison that can never be true

### A first look

Before any of the detail, here is the whole idea in a few lines. There is nothing to run yet: read
it, and read the output underneath it. Everything from Setup onward is where you start running
things.

```python
import pandas as pd

celsius = pd.Series([-12.0, None, None, -13.5])
print(celsius.isna().tolist())
print(celsius.mean(), celsius.interpolate().tolist())
```

```
[False, True, True, False]
-12.75 [-12.0, -12.5, -13.0, -13.5]
```

`None` in a list of decimals became `NaN`, and `isna` found both gaps. The mean of -12.75 is the
mean of the two real readings, because `mean` skipped the gaps. `interpolate` filled them with
values on a straight line between the readings on either side, two numbers that nobody measured.

## Setup

Two libraries, the module that catches warnings, and three helpers for building the readings.

- `pandas`, imported as `pd`, is the library, and the cell prints its version
- `numpy`, imported as `np`, provides `np.nan` for the comparisons in several cells, and adds up
  Svalbard's readings as an array once with `np.sum`
- `warnings` catches the warning in the last of the Common errors, so that it prints in one line
- `math`, `datetime` and `timedelta` build the readings

`readings` is the table of the four stations' hourly readings for 2025, 35,040 rows with the columns
`station`, `time` and `celsius`. The formula leaves Svalbard's 24 readings for 2 March as `None`,
which the table holds as `NaN`, and `stations` holds each station's latitude.

In [1]:
import math
import warnings
from datetime import datetime, timedelta

import numpy as np
import pandas as pd

STATIONS = {"Bergen": 8.0, "Oslo": 6.5, "Svalbard": -4.5, "Tromso": 3.5}   # each station's mean for the year
LATITUDES = {"Bergen": 60.39, "Oslo": 59.91, "Svalbard": 78.22, "Tromso": 69.65, "Kirkenes": 69.73}


def year_of_readings():
    """Every hour of 2025 at the four stations, as the sqlite3 guide makes them, with Svalbard silent on 2 March."""
    for n in range(365 * 24):
        hour = datetime(2025, 1, 1) + timedelta(hours=n)
        season = -math.cos(2 * math.pi * (n - 400) / (365 * 24))
        day = -math.cos(2 * math.pi * (hour.hour - 3) / 24)
        for i, (station, mean) in enumerate(STATIONS.items()):
            if station == "Svalbard" and hour.strftime("%Y-%m-%d") == "2025-03-02":
                celsius = None
            else:
                wobble = ((n * 37 + i * 101) % 17 - 8) / 10
                celsius = round(mean + 9 * season + 3 * day + wobble, 1) + 0.0
            yield station, hour.strftime("%Y-%m-%dT%H:%M"), celsius


readings = pd.DataFrame(list(year_of_readings()), columns=["station", "time", "celsius"])
readings["time"] = pd.to_datetime(readings["time"], format="%Y-%m-%dT%H:%M")
stations = pd.DataFrame({"station": list(LATITUDES), "latitude": list(LATITUDES.values())})

print("pandas", pd.__version__, "|", len(readings), "readings,", len(stations), "stations")

pandas 2.2.3 | 35040 readings, 5 stations


## Worked examples

### Finding the missing readings

`isna` gives `True` for every missing value, and `notna` for every value that is there:

In [2]:
missing = readings["celsius"].isna()

print("missing:", missing.sum(), "| present:", readings["celsius"].notna().sum())
readings[missing].head(3)

missing: 24 | present: 35016


,station,time,celsius
5762,Svalbard,2025-03-02 00:00:00,NaN
5766,Svalbard,2025-03-02 01:00:00,NaN
5770,Svalbard,2025-03-02 02:00:00,NaN


24 of the 35,040 readings are missing, and the first three are Svalbard's first three hours of 2
March. Grouping the mask by station shows that they all belong to Svalbard:

In [3]:
missing.groupby(readings["station"]).sum()

station
Bergen       0
Oslo         0
Svalbard    24
Tromso       0
Name: celsius, dtype: int64

On either side of the gap, Svalbard's readings are ordinary, as a Series indexed by time shows:

In [4]:
svalbard = readings[readings["station"] == "Svalbard"].set_index("time")["celsius"]

print(svalbard.loc["2025-03-01 22:00":"2025-03-02 01:00"])
print()
print(svalbard.loc["2025-03-02 22:00":"2025-03-03 01:00"])

time
2025-03-01 22:00:00   -11.6
2025-03-01 23:00:00   -12.0
2025-03-02 00:00:00     NaN
2025-03-02 01:00:00     NaN
Name: celsius, dtype: float64

time
2025-03-02 22:00:00     NaN
2025-03-02 23:00:00     NaN
2025-03-03 00:00:00   -13.5
2025-03-03 01:00:00   -13.7
Name: celsius, dtype: float64


The last reading before the gap was -12.0, at 23:00 on 1 March, and the first after it -13.5, at
midnight on 3 March. Those two numbers come back below, when the gap is filled.

### NaN, None and pd.NA

The same `None` becomes a different marker depending on the column's type:

In [5]:
print(pd.Series([1.5, None, 3.0]))
print()
print(pd.Series([1, None, 3]))
print()
print(pd.Series(["Oslo", None, "Bergen"]))
print()
print(pd.Series([1, None, 3], dtype="Int64"))

0    1.5
1    NaN
2    3.0
dtype: float64

0    1.0
1    NaN
2    3.0
dtype: float64

0      Oslo
1      None
2    Bergen
dtype: object

0       1
1    <NA>
2       3
dtype: Int64


Decimals keep `NaN`. Whole numbers with a gap turned into decimals, `float64`, so 1 became 1.0. Text
keeps `None` in a column of Python objects, `object`. The nullable `Int64` type kept the whole
numbers and marked the gap `<NA>`. `pd.isna` recognizes every one of these markers, and `==` finds
none of them in a Series:

In [6]:
print("isna:", pd.isna(np.nan), pd.isna(None), pd.isna(pd.NA))
print("np.nan == np.nan:", np.nan == np.nan, "| np.nan != np.nan:", np.nan != np.nan)
print("None == None:", None == None, "| in a Series:", (pd.Series(["Oslo", None]) == None).tolist())
print("pd.NA == pd.NA:", pd.NA == pd.NA)

isna: True True True
np.nan == np.nan: False | np.nan != np.nan: True
None == None: True | in a Series: [False, False]
pd.NA == pd.NA: <NA>


### What mean, sum and count do with the gaps

`mean` and `sum` skip missing values by default. `skipna=False` makes them include the gaps, and
then one missing value makes the answer missing:

In [7]:
print("mean:", round(svalbard.mean(), 2), "| with skipna=False:", svalbard.mean(skipna=False))
print("sum:", round(svalbard.sum(), 1), "| with skipna=False:", svalbard.sum(skipna=False))
print("sum as a NumPy array:", np.sum(svalbard.to_numpy()))
print("count:", svalbard.count(), "of", len(svalbard))

mean: -4.48 | with skipna=False: nan
sum: -39153.9 | with skipna=False: nan
sum as a NumPy array: nan
count: 8736 of 8760


The sum, -39153.9, is the total of the readings that are there, the same total that counting the
gaps as zero would give. The same values as a NumPy array add up to `nan`, because NumPy's own sum
skips nothing. `count` counts the values that are there, 8,736 of Svalbard's 8,760 hours, and the
mean of -4.48 is the mean of those 8,736, which is usually the number to report, along with how many
readings it was taken over.

### Dropping rows with dropna

`dropna` on the readings removes the 24 rows with no temperature:

In [8]:
print(readings.shape, "->", readings.dropna().shape)

(35040, 3) -> (35016, 3)


A table with gaps in several columns shows the options. This small logbook has a station, a
temperature and a wind speed, and each row is missing something different:

In [9]:
log = pd.DataFrame({
    "station": ["Oslo", "Oslo", "Bergen", None],
    "celsius": [1.2, None, 2.5, None],
    "wind": [None, None, 3.0, None],
})

print(log)
print("\nhow='any', the default:\n", log.dropna(), sep="")
print("\nhow='all':\n", log.dropna(how="all"), sep="")
print("\nsubset=['celsius']:\n", log.dropna(subset=["celsius"]), sep="")
print("\nthresh=2:\n", log.dropna(thresh=2), sep="")

  station  celsius  wind
0    Oslo      1.2   NaN
1    Oslo      NaN   NaN
2  Bergen      2.5   3.0
3    None      NaN   NaN

how='any', the default:
  station  celsius  wind
2  Bergen      2.5   3.0

how='all':
  station  celsius  wind
0    Oslo      1.2   NaN
1    Oslo      NaN   NaN
2  Bergen      2.5   3.0

subset=['celsius']:
  station  celsius  wind
0    Oslo      1.2   NaN
2  Bergen      2.5   3.0

thresh=2:
  station  celsius  wind
0    Oslo      1.2   NaN
2  Bergen      2.5   3.0


The default drops a row with any missing value, and only Bergen's row is complete. `how="all"` drops
only row 3, which has nothing at all. `subset` looks at the columns named, so the two rows with a
temperature survive. `thresh=2` keeps a row with at least two values, here rows 0 and 2. Dropping by
the column you are about to use, with `subset`, removes only the rows that lack it.

### Filling with a value

`fillna` puts one value in every gap. Filling Svalbard's silent day with 0 makes it a day of 0
degrees, in a month whose real mean was about -9:

In [10]:
filled_with_zero = svalbard.fillna(0)

print("March, gaps left out:", round(svalbard.loc["2025-03"].mean(), 2))
print("March, gaps as 0:    ", round(filled_with_zero.loc["2025-03"].mean(), 2))

March, gaps left out: -9.28
March, gaps as 0:     -8.98


The fill made Svalbard's March 0.3 degrees warmer, from 24 readings nobody took. A fill value has to
mean something for the data, and 0 degrees means nothing here.

### Filling from neighbors with ffill, bfill and interpolate

`ffill` copies the last reading forward into the gap, `bfill` copies the next one back, and
`interpolate` draws a straight line between them:

In [11]:
fills = pd.DataFrame({
    "ffill": svalbard.ffill(),
    "bfill": svalbard.bfill(),
    "interpolate": svalbard.interpolate(),
})

print(fills.loc["2025-03-02 00:00":"2025-03-02 02:00"])
print()
print("mean for 2 March:")
print(fills.loc["2025-03-02"].mean().round(2))

                     ffill  bfill  interpolate
time                                          
2025-03-02 00:00:00  -12.0  -13.5       -12.06
2025-03-02 01:00:00  -12.0  -13.5       -12.12
2025-03-02 02:00:00  -12.0  -13.5       -12.18

mean for 2 March:
ffill         -12.00
bfill         -13.50
interpolate   -12.75
dtype: float64


`ffill` filled the whole day with -12.0, the reading at 23:00 the night before, and `bfill` with
-13.5, the one at midnight after. `interpolate` stepped from one to the other by 0.06 degrees an
hour. The three fills give three means for the same day, -12.0, -13.5 and -12.75, and none of them
is a measurement. `limit` caps how far a fill reaches, so `svalbard.ffill(limit=3)` fills three
hours and leaves 21 of the gap missing:

In [12]:
print("still missing after ffill(limit=3):", svalbard.ffill(limit=3).isna().sum())

still missing after ffill(limit=3): 21


### Keeping whole numbers whole with Int64

`Int64`, with a capital I, is pandas' nullable whole-number type. It holds `pd.NA` and keeps the
other values as integers, and its reductions skip the gaps as the decimal ones do:

In [13]:
visits = pd.Series([4, None, 7, 5], dtype="Int64")

print(visits)
print("sum:", visits.sum(), "| mean:", round(visits.mean(), 2))

0       4
1    <NA>
2       7
3       5
dtype: Int64
sum: 16 | mean: 5.33


## Your turn

Five tasks. Write your answer in the cell under each task and run it.

Try a task before you look at its answer. Reading a solution teaches you much less than
getting there yourself, even slowly.

When you are ready: [**open the solutions
notebook**](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/10-missing-data-solutions.ipynb).

**1.** Find the first and last hour of Svalbard's gap from the index of its missing readings.

In [14]:
# your code here

**2.** Print Svalbard's mean for March with the gaps left out, and with `skipna=False`.

In [15]:
# your code here

**3.** Fill Svalbard's gap with `interpolate`, and print the filled readings at 06:00, 12:00 and
18:00 on 2 March.

In [16]:
# your code here

**4.** Rebuild the small logbook, and keep only the rows that have a wind speed, using `dropna` with
`subset`.

In [17]:
# your code here

**5.** Make a Series of the whole numbers 4, 6 and 9 with a gap after the 6, as `Int64`, and print
its sum and its mean.

In [18]:
# your code here

## Common errors

### No error, and 0 missing readings found with == np.nan

In [19]:
print("== np.nan:", (readings["celsius"] == np.nan).sum())
print("isna():   ", readings["celsius"].isna().sum())

== np.nan: 0
isna():    24


`NaN` is not equal to anything, itself included, so `== np.nan` is `False` for every row and the
count is 0. The 24 missing readings are there, and `isna` finds them. Use `isna` or `notna` whenever
the question is whether a value is missing.

### IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer

In [20]:
readings["celsius"].round().astype(int)

IntCastingNaNError: Cannot convert non-finite values (NA or inf) to integer

Rounding gave whole numbers, and `astype(int)` asks for NumPy's integer type, which has no way to
mark a gap, so the 24 `NaN` values stop the conversion. "Non-finite" covers `NaN` and infinity.
Decide what the gap should be first, with `dropna` or a fill, or keep it as a gap with the nullable
type:

In [21]:
readings["celsius"].round().astype("Int64").loc[5760:5763]

5760      -1
5761      -3
5762    <NA>
5763      -4
Name: celsius, dtype: Int64

Row 5762, Svalbard at midnight on 2 March, is `<NA>`, and the rows around it are whole numbers.

### TypeError: boolean value of NA is ambiguous

In [22]:
print("np.nan > 3:", np.nan > 3)

visits = pd.Series([4, None, 7, 5], dtype="Int64")
if visits.iloc[1] > 3:
    print("a busy day")

np.nan > 3: False


TypeError: boolean value of NA is ambiguous

`visits.iloc[1]` is `pd.NA`, and `pd.NA > 3` is `<NA>` too, because a missing number may or may not
be greater than 3. `if` needs `True` or `False`, and pandas raises instead of choosing one. Ask the
question that has an answer first: `if pd.notna(visits.iloc[1]) and visits.iloc[1] > 3:`. A `NaN` in
a decimal column behaves differently, and the cell's first line shows that `np.nan > 3` is `False`.

### FutureWarning: Series.fillna with 'method' is deprecated and will raise in a future version. Use obj.ffill() or obj.bfill() instead.

In [23]:
with warnings.catch_warnings(record=True) as caught:
    warnings.simplefilter("always")
    old_style = svalbard.fillna(method="ffill")

for warning in caught:
    print(f"{warning.category.__name__}: {warning.message}")
print("still missing:", old_style.isna().sum())
print("same as ffill():", old_style.equals(svalbard.ffill()))

still missing: 0
same as ffill(): True


`fillna(method="ffill")` is how older code and many answers online forward-fill, and pandas 2.2
still does it, warning that a later version will raise instead. The cell catches the warning and
prints it in one line. `svalbard.ffill()` does the same fill with no warning.

## Recap

- Find and count missing values with `isna` and `notna`, and never with `== np.nan`
- Tell `NaN`, `None` and `pd.NA` apart, and keep whole numbers whole with `Int64`
- Know that `mean` and `sum` skip gaps unless given `skipna=False`, and that `count` counts only the
  values that are there
- Drop rows with `dropna`, choosing the columns that matter with `subset`
- Fill gaps with a value, `ffill`, `bfill` or `interpolate`, and say that the numbers were filled

## What is next

The **Adding and Changing Columns** notebook writes new values into a table, with `assign` and
`loc`, and explains the `SettingWithCopyWarning` that a filtered table can raise.

---

&#8592; **Previous:** [The Index](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/09-the-index.ipynb)  &nbsp;·&nbsp;  [Pandas, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pandas-deep-dive.html)  &nbsp;·&nbsp;  **Next:** [Adding and Changing Columns](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pandas-deep-dive/11-adding-and-changing-columns.ipynb) &#8594;
